# GLiClass 퓨샷 파인튜닝 시범 실행 (edge, k = 1 / 3 / 5)

**목적**: 실제 GLiClass edge 모델을 우리 로그로 퓨샷 파인튜닝했을 때 파이프라인이 돌아가는지, 점수가 어느 방향으로 움직이는지 처음으로 확인한다. 본실험이 아니라 **시범**이다.

**무엇을 재나** (파인튜닝 전 = 기준선, k = 1, 3, 5 각각 시드 3번)

| 시험 | 확인하는 것 | 성공이면 |
|---|---|---|
| A. 학습한 전술 정확도 | 공부한 효과 | 오름 |
| B. 숨긴 전술 정확도 (defense-evasion, exfiltration, discovery) | 처음 보는 공격도 맞히나 (제로샷) | 오르거나 유지 |
| C. ag_news 정확도 (일반 뉴스 4분류, 500개) | 원래 실력이 남았나 | 거의 그대로 |
| D. 판별력 = 공격 재현율 - 오탐률 | 공격과 정상을 구분하나 | 0보다 커짐 |

**설정**: 학습은 GLiClass 공식 `train.py` 그대로 사용 (multi-label, sigmoid + focal loss α 0.5 γ 2, AdamW). 숨긴 전술은 학습 데이터에서도, 학습 때 라벨 목록에서도 완전히 뺀다.

**한계 (회의에서 같이 말할 것)**: 라벨이 예전 버전(MITRE 공식 전술 설명 11개), 데이터 3분할 없음, 망각 방지(리플레이/EWC) 미적용, k=8/16은 DataDog 예시가 부족해 제외.

**준비**: 런타임 > 런타임 유형 변경 > **T4 GPU**. 왼쪽 파일 탭에 `cloudtrail_windows.csv` 업로드. 전체 실행 시간 약 20~30분.

## 1. 설치 (GLiClass 공식 코드, 확인한 버전으로 고정)

In [ ]:
COMMIT = '6a00f8067648f25aa0f4f477d167c5f391cb4165'
!pip -q install "git+https://github.com/Knowledgator/GLiClass.git@{COMMIT}"
!git clone -q https://github.com/Knowledgator/GLiClass.git gliclass_repo && cd gliclass_repo && git checkout -q {COMMIT}
!ls gliclass_repo/train.py

설치 뒤 다음 칸에서 import 에러가 나면: 런타임 > 세션 다시 시작 후 **2번부터** 다시 실행 (1번은 다시 안 해도 됨).

## 2. 설정

In [ ]:
import os, json, random, subprocess, shutil, time
import numpy as np, pandas as pd, torch

MODEL = 'knowledgator/gliclass-edge-v3.0'
K_LIST = [1, 3, 5]
SEEDS = [0, 1, 2]
LR = 2e-5          # train.py 기본 encoder_lr
EPOCHS = 10        # 예시가 몇십 줄뿐이라 3에폭이면 거의 안 움직여서 시범은 10으로
BATCH = 8
MAX_LEN = 2048     # 실제 입력 약 1,300토큰
BENIGN_POOL_N, BENIGN_TEST_N = 50, 500
HELD_OUT = ['defense-evasion', 'exfiltration', 'discovery']   # 학습에서 완전히 숨기는 전술

LABELS = {   # 사전 실험과 같은 라벨 문장 (MITRE 공식 전술 설명 + benign)
    'initial-access': 'The adversary is trying to get into your network.',
    'execution': 'The adversary is trying to run malicious code.',
    'persistence': 'The adversary is trying to maintain their foothold.',
    'privilege-escalation': 'The adversary is trying to gain higher-level permissions.',
    'defense-evasion': 'The adversary is trying to avoid being detected.',
    'credential-access': 'The adversary is trying to steal account names and passwords.',
    'discovery': 'The adversary is trying to figure out your environment.',
    'lateral-movement': 'The adversary is trying to move through your environment.',
    'exfiltration': 'The adversary is trying to steal data.',
    'impact': 'The adversary is trying to manipulate, interrupt, or destroy your systems and data.',
    'benign': 'Normal administrative, automated or user activity with no adversary.',
}
TEXT2LABEL = {v: k for k, v in LABELS.items()}
SEEN = [t for t in LABELS if t not in HELD_OUT]            # 학습에 쓰는 라벨 (benign 포함)
print('GPU:', torch.cuda.is_available(), '| 학습 라벨:', SEEN)

## 3. 데이터 나누기

예시 후보 = DataDog 계정 로그(숨긴 전술 제외), 시험 = invictus 계정 로그. 정상 예시 50개는 시험과 겹치지 않게 따로 뺀다.

In [ ]:
df = pd.read_csv('cloudtrail_windows.csv').drop_duplicates(['source', 'text', 'labels']).reset_index(drop=True)
df['label'] = df['labels'].str.split('|').str[0]
pool = df[(df.source == 'datadog') & (~df.label.isin(HELD_OUT))]
inv = df[df.source == 'invictus']
benign = inv[inv.label == 'benign'].sample(frac=1, random_state=42)
benign_pool = benign.iloc[:BENIGN_POOL_N]
test = pd.concat([inv[inv.label != 'benign'], benign.iloc[BENIGN_POOL_N:BENIGN_POOL_N + BENIGN_TEST_N]])
test = test[~test.text.isin(df[df.source == 'datadog'].text)].reset_index(drop=True)

print('예시 후보(전술별):', pool.label.value_counts().to_dict())
print('시험(전술별):', test.label.value_counts().to_dict())
print('숨긴 전술이 학습 후보에 섞였나:', pool.label.isin(HELD_OUT).any())

## 4. 학습 데이터 만들기

라벨마다 k개. 같은 공격 실행(run)에서 겹치는 윈도우만 뽑히지 않게 **서로 다른 실행부터 하나씩** 고른다. 형식은 GLiClass 공식 형식 `{text, all_labels, true_labels}`.

In [ ]:
def pick(group, k, rng):
    runs = list(group.groupby('run_id'))
    rng.shuffle(runs)
    picked, i = [], 0
    while len(picked) < min(k, len(group)):
        rid, g = runs[i % len(runs)]
        rest = g[~g.index.isin([p.name for p in picked])]
        if len(rest):
            picked.append(rest.sample(1, random_state=rng.randint(0, 10**6)).iloc[0])
        i += 1
    return picked

def make_train(k, seed):
    rng = random.Random(seed)
    seen_texts = [LABELS[t] for t in SEEN]             # 숨긴 전술 문장은 라벨 목록에도 없음
    rows = []
    for tactic, g in pool.groupby('label'):
        for r in pick(g, k, rng):
            rows.append({'text': r.text, 'all_labels': seen_texts, 'true_labels': [LABELS[tactic]]})
    for _, r in benign_pool.sample(k, random_state=seed).iterrows():
        rows.append({'text': r.text, 'all_labels': seen_texts, 'true_labels': [LABELS['benign']]})
    rng.shuffle(rows)
    return rows

demo = make_train(1, 0)
print(len(demo), '줄 / 예시 한 줄:')
print(json.dumps(demo[0], indent=1)[:600])

## 5. 평가 함수

시험 때는 숨긴 전술 포함 **라벨 11개 전부**를 넣고 1등 라벨을 예측으로 쓴다.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from gliclass import GLiClassModel, ZeroShotClassificationPipeline

agnews = load_dataset('fancyzhx/ag_news', split='test').shuffle(seed=0).select(range(500))
AG_LABELS = ['World', 'Sports', 'Business', 'Sci/Tech']

def top1(pipe, texts, labels):
    out = pipe(texts, labels, threshold=0.0, batch_size=16)
    return [max(r, key=lambda x: x['score'])['label'] for r in out]

def evaluate(path):
    model = GLiClassModel.from_pretrained(path)
    tok = AutoTokenizer.from_pretrained(path)
    pipe = ZeroShotClassificationPipeline(model, tok, classification_type='multi-label',
                                          device='cuda:0', max_length=MAX_LEN, progress_bar=False)
    y = test.label.tolist()
    p = [TEXT2LABEL[t] for t in top1(pipe, test.text.tolist(), list(LABELS.values()))]
    att = [t != 'benign' for t in y]
    recall = np.mean([q != 'benign' for q, a in zip(p, att) if a])
    fpr = np.mean([q != 'benign' for q, a in zip(p, att) if not a])
    seen = [q == t for q, t in zip(p, y) if t != 'benign' and t not in HELD_OUT]
    held = [q == t for q, t in zip(p, y) if t in HELD_OUT]
    ag = top1(pipe, agnews['text'], AG_LABELS)
    ag_acc = np.mean([AG_LABELS.index(a) == b for a, b in zip(ag, agnews['label'])])
    del model, pipe; torch.cuda.empty_cache()
    return dict(seen_acc=np.mean(seen), heldout_acc=np.mean(held), agnews_acc=ag_acc,
                attack_recall=recall, fpr=fpr, separation=recall - fpr,
                pred_dist=pd.Series(p).value_counts().to_dict())

t0 = time.time()
base = evaluate(MODEL)
print(f'기준선(파인튜닝 전) {time.time()-t0:.0f}초')
{k: (round(v, 3) if isinstance(v, float) else v) for k, v in base.items()}

## 6. 퓨샷 파인튜닝 + 평가 (k = 1, 3, 5 × 시드 3)

GLiClass 공식 `train.py`를 그대로 호출한다. 한 번 돌 때마다 결과를 `pilot_results.csv`에 저장하므로 중간에 끊겨도 남는다.

In [ ]:
arch = GLiClassModel.from_pretrained(MODEL).config.architecture_type
results = [{'k': 0, 'seed': 0, **{k: v for k, v in base.items() if k != 'pred_dist'}}]

for k in K_LIST:
    for seed in SEEDS:
        rows = make_train(k, seed)
        json.dump(rows, open('train.json', 'w'))
        out = f'runs/k{k}_s{seed}'
        cmd = ['python', 'gliclass_repo/train.py', '--model_name', MODEL,
               '--data_path', 'train.json', '--val_data_path', 'train.json',   # 검증은 끔(save_steps 크게) - 90/10 분할로 데이터 잃지 않게
               '--save_path', out, '--problem_type', 'multi_label_classification',
               '--architecture_type', arch,
               '--encoder_lr', str(LR), '--others_lr', str(LR),
               '--num_epochs', str(EPOCHS), '--batch_size', str(BATCH), '--max_length', str(MAX_LEN),
               '--focal_loss_alpha', '0.5', '--focal_loss_gamma', '2',
               '--num_workers', '0', '--save_steps', '100000', '--logging_steps', '5']
        t0 = time.time()
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-2000:], r.stderr[-3000:]); raise SystemExit(f'학습 실패 k={k} seed={seed}')
        m = evaluate(os.path.join(out, 'final_model'))
        shutil.rmtree(out, ignore_errors=True)
        results.append({'k': k, 'seed': seed, 'n_train': len(rows), **{a: b for a, b in m.items() if a != 'pred_dist'}})
        pd.DataFrame(results).to_csv('pilot_results.csv', index=False)
        print(f'k={k} seed={seed} ({len(rows)}줄, {time.time()-t0:.0f}초) '
              f'학습전술 {m["seen_acc"]:.3f} | 숨긴전술 {m["heldout_acc"]:.3f} | ag_news {m["agnews_acc"]:.3f} | '
              f'판별력 {m["separation"]:.3f} | 예측분포 {m["pred_dist"]}')

## 7. 요약

In [ ]:
res = pd.read_csv('pilot_results.csv')
cols = ['seen_acc', 'heldout_acc', 'agnews_acc', 'attack_recall', 'fpr', 'separation']
summary = res.groupby('k')[cols].agg(['mean', 'std']).round(3)
summary

In [ ]:
import matplotlib.pyplot as plt
m, s = res.groupby('k')[cols].mean(), res.groupby('k')[cols].std().fillna(0)
names = {'seen_acc': 'A. seen tactics acc', 'heldout_acc': 'B. held-out tactics acc',
         'agnews_acc': 'C. ag_news acc (original skill)', 'separation': 'D. recall - FPR'}
fig, ax = plt.subplots(figsize=(7, 4))
for c, n in names.items():
    ax.errorbar(m.index, m[c], yerr=s[c], marker='o', capsize=3, label=n)
ax.axhline(0, color='grey', lw=.8)
ax.set(xlabel='examples per label (k), 0 = before fine-tuning', ylabel='score', xticks=[0] + K_LIST, ylim=(-0.3, 1.05),
       title='GLiClass edge few-shot fine-tuning (pilot)')
ax.legend(fontsize=8); ax.grid(alpha=.3); fig.tight_layout(); fig.savefig('pilot_curve.png', dpi=200)

## 8. 결과 읽는 법

| 결과 | 판정 |
|---|---|
| A 오름, B 오르거나 유지, C 거의 그대로 | 가설 방향과 일치 (시범 수준) |
| A 오름, B 그대로 또는 하락 | 학습한 전술만 외움 → 지도학습처럼 됨 |
| A 오름, C 크게 하락 | 원래 실력 망각 → 리플레이/EWC 필요 |
| 아무것도 안 움직임 | 학습량 부족 → EPOCHS 또는 LR 조정 |
| 예측분포가 한 라벨로 몰림 | 사전 실험과 같은 쏠림, 판별력 0 근처 |

`pilot_results.csv`와 `pilot_curve.png`를 회의에 가져가면 된다.